# GymLeader: SFT on Google Colab (setup only)

This notebook runs on **Google Colab with a GPU** (Runtime → Change runtime type → GPU), not on the Mac.
It follows Unsloth's Qwen3.5 fine-tuning guide: https://unsloth.ai/docs/models/qwen3.5/fine-tune

For now it only prepares everything and checks it: install, GPU, Drive, data, model, chat template.
No LoRA and no training yet.

## 1. Install

Unsloth's guide says to upgrade `unsloth` and `unsloth_zoo` with a forced, no-cache reinstall.
Qwen3.5 **needs transformers v5** (older versions can't load it), and Colab ships v4, so we ask for v5 in the same command:
pip then picks versions that work together, or fails loudly instead of silently mixing.

This takes a few minutes. If the versions printed below look old, use Runtime → Restart session and run from the next cell.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo "transformers>=5,<6"

In [ ]:
import unsloth  # import first: Unsloth patches transformers/trl when it is imported
import torch, transformers, trl

for lib in (unsloth, transformers, trl, torch):
    print(f"{lib.__name__:13} {lib.__version__}")
assert transformers.__version__.startswith("5."), "Qwen3.5 needs transformers v5: restart the session and re-run"

## 2. GPU check

Training speed and memory depend on the GPU. bf16 matters because we load the model in 16-bit (Unsloth advises against 4-bit for Qwen3.5);
GPUs without bf16 fall back to fp16. Qwen3.5 has custom (Mamba-style) Triton kernels that are compiled the first time they run,
which is especially slow on a T4.

In [ ]:
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU:    {gpu.name}")
print(f"memory: {gpu.total_memory / 1e9:.1f} GB")
print(f"bf16:   {torch.cuda.is_bf16_supported()}")
if "T4" in gpu.name:
    print("Note: T4 detected. Qwen3.5's kernels compile slowly on T4, so the first steps will be slow. That is expected.")

## 3. Google Drive

Colab deletes its disk when the session ends. Everything we want to keep (adapters, logs) goes to Drive,
in one folder for this project.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/SFT-Gym-Workout-Program")
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## 4. Data

We clone the repo at `main`, because we need today's code too: at the tag `data-v1`, `src/evaluate.py` doesn't exist yet,
and section 6 imports its prompt builder. The *data*, though, must be the frozen dataset we tagged as `data-v1`.
So we fetch the tags and assert that `data/processed/` on `main` is byte-for-byte the same as at `data-v1`
(`git diff --quiet` exits with 0 only when there are no differences). If someone regenerates the data, this cell stops.

Only `train.jsonl` and `val.jsonl` are loaded. **`test.jsonl` is never loaded here**: the test set is only for the final
evaluation on the Mac, and looking at it during training would make the test score meaningless.

In [ ]:
import json, subprocess, sys

REPO = Path("/content/SFT-Gym-Workout-Program")
if not REPO.exists():
    !git clone --quiet --branch main https://github.com/riccardoroselli/SFT-Gym-Workout-Program.git {REPO}
!git -C {REPO} fetch --quiet --tags
!git -C {REPO} log -1 --oneline
diff = subprocess.run(["git", "-C", str(REPO), "diff", "--quiet", "data-v1", "HEAD", "--", "data/processed/"])
assert diff.returncode == 0, "data/processed/ on main differs from data-v1: the training data is not the frozen dataset"
print("data/processed/ matches data-v1")
sys.path.insert(0, str(REPO / "src"))  # so section 6 can import evaluate.py

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

train = read_jsonl(REPO / "data/processed/train.jsonl")
val = read_jsonl(REPO / "data/processed/val.jsonl")
assert (len(train), len(val)) == (2000, 200), (len(train), len(val))
print(f"train {len(train)}, val {len(val)}")
print(train[0]["messages"])

## 5. Model

Settings from Unsloth's guide: 16-bit weights (bf16 LoRA, not 4-bit QLoRA, which loses more accuracy on Qwen3.5) and
`full_finetuning=False`, because we will train a small LoRA adapter instead of all 2B weights.
`max_seq_length=1024` is plenty: our longest training row is far shorter (checked in the next section).
Unsloth's table says the 2B model needs about 5 GB for bf16 LoRA.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-2B",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

## 6. Chat template check

The model must be trained on **exactly** the text it will see at evaluation time. `src/evaluate.py` builds the prompt
with `build_prompt()`, which we import here instead of copying, so the two can't drift apart. It ends with an empty
`<think>\n\n</think>\n\n` block (thinking off). Importing `evaluate.py` works on Colab because mlx (Mac only) is
imported only inside its `generate` step.
A training row is the whole conversation formatted the same way, so it must be that prompt, then the assistant JSON,
then the end-of-turn token `<|im_end|>`. If Unsloth's tokenizer formatted it differently, the fine-tuned model would
learn one format and be tested on another.

The reference prompt uses the plain Hugging Face tokenizer, the same one `evaluate.py` gets through mlx-lm.
Qwen3.5 is a vision-language model, so Unsloth may return a processor: we use the text tokenizer inside it.

In [ ]:
from transformers import AutoTokenizer
from evaluate import build_prompt

text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # processor -> its text tokenizer
reference = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B")

def train_text(row):
    return text_tokenizer.apply_chat_template(row["messages"], tokenize=False, enable_thinking=False)

row = train[0]
system, user, assistant = row["messages"]
text = train_text(row)
print(text)

eval_prompt = build_prompt(reference, [system, user])  # what evaluate.py sends, with no few-shot turns
END = "<|im_end|>"
assert text.startswith(eval_prompt), "training text does not start with the evaluate.py prompt"
assert text.rstrip("\n").endswith(assistant["content"] + END), "training text does not end with answer + <|im_end|>"
assert text == eval_prompt + assistant["content"] + END + "\n", "unexpected extra text"
print("template check passed: training text = evaluate.py prompt + answer JSON + <|im_end|>")

In [ ]:
lengths = [len(text_tokenizer(train_text(r))["input_ids"]) for r in train]
longest = max(range(len(train)), key=lengths.__getitem__)
print(f"longest train row: {lengths[longest]} tokens (row {longest}), max_seq_length {MAX_SEQ_LENGTH}")
assert lengths[longest] <= MAX_SEQ_LENGTH